# Series de precios de las acciones

In [ ]:
pip install tidyfinance polars curl_cffi

In [ ]:
import polars as pl

import io
import re
import zipfile
from curl_cffi import requests

import tidyfinance as tf
#tf.set_backend("polars")


# Datos financieros gratuitos

In [ ]:
# Nos bajamos los datos directamente de Yahoo Finance

import yfinance as yf

asset="AAPL"

prices = yf.download(
    asset,
    start = "2000-01-01",
    end = "2024-12-31",
    auto_adjust=True).dropna()

prices.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2000-01-03,0.837002,0.841208,0.760359,0.784193,535796800
2000-01-04,0.766435,0.827188,0.756621,0.809430,512377600
2000-01-05,0.777650,0.826721,0.770173,0.775781,778321600
2000-01-06,0.710354,0.800083,0.710354,0.793540,767972800
2000-01-07,0.744002,0.755219,0.714093,0.721570,460734400


In [ ]:
# Nos bajamos los datos con la herramienta de Tidy Finance de Yahoo Finance.

prices = tf.download_data(
    domain="Stock Prices",
    symbols="AAPL",
    start_date="2000-01-01",
    end_date="2024-12-31"
)
prices.head()

,symbol,date,volume,open,low,high,close,adjusted_close
0,AAPL,2000-01-03,535796800,0.936384,0.907924,1.004464,0.999442,0.837002
1,AAPL,2000-01-04,512377600,0.966518,0.903460,0.987723,0.915179,0.766435
2,AAPL,2000-01-05,778321600,0.926339,0.919643,0.987165,0.928571,0.777650
3,AAPL,2000-01-06,767972800,0.947545,0.848214,0.955357,0.848214,0.710353
4,AAPL,2000-01-07,460734400,0.861607,0.852679,0.901786,0.888393,0.744002


#Dataframe de artículos científicos

## Fama y French (1993)
Este código permite descargar los factores de Fama y French desde la página de Kenneth French, descomprimir el archivo ZIP y extraer su contenido como texto, sin necesidad de guardar ningún archivo en tu ordenador.

In [ ]:
start_date = "1960-01-01"
end_date = "2024-12-31"

In [ ]:
dataset = "F-F_Research_Data_Factors"
base_url = "http://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
url = f"{base_url}{dataset}_CSV.zip"

In [ ]:
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    file_name = zf.namelist()[0]  # Ken French ZIPs contain one file
    raw_text = zf.read(file_name).decode("latin1")

In [ ]:
chunks = raw_text.split("\r\n\r\n")
table_text = max(chunks, key=len)

In [ ]:
match = re.search(r"^\s*,", table_text, flags=re.M)
start = match.start()
csv_text = "Date" + table_text[start:]

factors_ff3_monthly_raw = pl.read_csv(csv_text.encode("latin1")).with_columns(
    pl.exclude("Date").cast(pl.String).str.strip_chars().cast(pl.Float64)
)

In [ ]:
s = factors_ff3_monthly_raw["Date"].cast(pl.String)
n = s.str.len_chars()

if (n == 8).all():  # daily: YYYYMMDD
    date = s.str.to_date("%Y%m%d")
elif (n == 6).all():  # monthly: YYYYMM
    date = (s + "01").str.to_date("%Y%m%d")
elif (n == 4).all():  # annual: YYYY, assigned to year-end
    date = (s + "1231").str.to_date("%Y%m%d")
else:
    raise ValueError("Unknown date format in Fama–French index.")

factors_ff3_monthly_raw = factors_ff3_monthly_raw.with_columns(date=date).drop("Date")

In [ ]:
if start_date:
    factors_ff3_monthly_raw = factors_ff3_monthly_raw.filter(
        pl.col("date") >= pl.lit(start_date).str.to_date()
    )
if end_date:
    factors_ff3_monthly_raw = factors_ff3_monthly_raw.filter(
        pl.col("date") <= pl.lit(end_date).str.to_date()
    )

factors_ff3_monthly = (
    factors_ff3_monthly_raw.rename(lambda c: c.lower())
    .rename({"mkt-rf": "mkt_excess", "rf": "risk_free"})
    .with_columns(pl.exclude("date").replace({-99.99: None, -999: None}) / 100)
    .select(pl.col("date"), pl.exclude("date"))
)
factors_ff3_monthly

date,mkt_excess,smb,hml,risk_free
date,f64,f64,f64,f64
1960-01-01,-0.0698,0.0212,0.0265,0.0033
1960-02-01,0.0116,0.006,-0.0197,0.0029
1960-03-01,-0.0163,-0.0055,-0.0275,0.0035
1960-04-01,-0.0171,0.0022,-0.0214,0.0019
1960-05-01,0.0312,0.0129,-0.0373,0.0027
…,…,…,…,…
2024-08-01,0.016,-0.0349,-0.0113,0.0048
2024-09-01,0.0172,-0.0013,-0.0273,0.004
2024-10-01,-0.01,-0.0095,0.0091,0.0039


### Utilizando la libreria tidyfinance

In [ ]:
factors_ff3_monthly = tf.download_data(
    domain="Fama-French",
    dataset="Fama/French 3 Factors",
    start_date=start_date,
    end_date=end_date,
)
factors_ff3_monthly

,date,mkt_excess,smb,hml,risk_free
0,1960-01-01,-0.0698,0.0212,0.0265,0.0033
1,1960-02-01,0.0116,0.0060,-0.0197,0.0029
2,1960-03-01,-0.0163,-0.0055,-0.0275,0.0035
3,1960-04-01,-0.0171,0.0022,-0.0214,0.0019
4,1960-05-01,0.0312,0.0129,-0.0373,0.0027
...,...,...,...,...,...
775,2024-08-01,0.0160,-0.0349,-0.0113,0.0048
776,2024-09-01,0.0172,-0.0013,-0.0273,0.0040
777,2024-10-01,-0.0100,-0.0095,0.0091,0.0039
778,2024-11-01,0.0649,0.0449,0.0038,0.0040


In [ ]:
factors_ff5_monthly = tf.download_data(
    domain="Fama-French",
    dataset="Fama/French 5 Factors (2x3)",
    start_date=start_date,
    end_date=end_date,
)
factors_ff5_monthly

,date,mkt_excess,smb,hml,rmw,cma,risk_free
0,1963-07-01,-0.0039,-0.0048,-0.0084,0.0064,-0.0115,0.0027
1,1963-08-01,0.0508,-0.0080,0.0172,0.0040,-0.0038,0.0025
2,1963-09-01,-0.0157,-0.0043,-0.0002,-0.0078,0.0015,0.0027
3,1963-10-01,0.0254,-0.0134,-0.0003,0.0279,-0.0225,0.0029
4,1963-11-01,-0.0086,-0.0085,0.0178,-0.0043,0.0227,0.0027
...,...,...,...,...,...,...,...
733,2024-08-01,0.0161,-0.0357,-0.0113,0.0075,0.0079,0.0048
734,2024-09-01,0.0173,-0.0091,-0.0273,0.0019,-0.0025,0.0040
735,2024-10-01,-0.0100,-0.0086,0.0091,-0.0149,0.0102,0.0039
736,2024-11-01,0.0649,0.0463,0.0038,-0.0227,-0.0193,0.0040


In [ ]:
tf.get_available_famafrench_datasets()

['F-F_Research_Data_Factors',
 'F-F_Research_Data_Factors_weekly',
 'F-F_Research_Data_Factors_daily',
 'F-F_Research_Data_5_Factors_2x3',
 'F-F_Research_Data_5_Factors_2x3_daily',
 'Portfolios_Formed_on_ME',
 'Portfolios_Formed_on_ME_Wout_Div',
 'Portfolios_Formed_on_ME_Daily',
 'Portfolios_Formed_on_BE-ME',
 'Portfolios_Formed_on_BE-ME_Wout_Div',
 'Portfolios_Formed_on_BE-ME_Daily',
 'Portfolios_Formed_on_OP',
 'Portfolios_Formed_on_OP_Wout_Div',
 'Portfolios_Formed_on_OP_Daily',
 'Portfolios_Formed_on_INV',
 'Portfolios_Formed_on_INV_Wout_Div',
 'Portfolios_Formed_on_INV_Daily',
 '6_Portfolios_2x3',
 '6_Portfolios_2x3_Wout_Div',
 '6_Portfolios_2x3_weekly',
 '6_Portfolios_2x3_daily',
 '25_Portfolios_5x5',
 '25_Portfolios_5x5_Wout_Div',
 '25_Portfolios_5x5_Daily',
 '100_Portfolios_10x10',
 '100_Portfolios_10x10_Wout_Div',
 '100_Portfolios_10x10_Daily',
 '6_Portfolios_ME_OP_2x3',
 '6_Portfolios_ME_OP_2x3_Wout_Div',
 '6_Portfolios_ME_OP_2x3_daily',
 '25_Portfolios_ME_OP_5x5',
 '25_Portf

## q-factor (Hou et al. 2015)

In [ ]:
factors_q_monthly_link = (
    "https://global-q.org/uploads/1/2/2/6/122679606/q5_factors_monthly_2024.csv"
)

factors_q_monthly = (
    pl.read_csv(requests.get(factors_q_monthly_link).content)
    .with_columns(
        date=pl.date(pl.col("year"), pl.col("month"), 1)
    )
    .drop("year", "month")
    .rename(lambda x: x.replace("R_", "").lower())
    .rename({"f": "risk_free", "mkt": "mkt_excess"})
    .filter(
        (pl.col("date") >= pl.lit(start_date).str.to_date())
        & (pl.col("date") <= pl.lit(end_date).str.to_date())
    )
    .with_columns(pl.exclude("date") / 100)
    .select("date", "risk_free", "mkt_excess", "me", "ia", "roe", "eg")
)

### Utilizando la libreria tidyfinance

In [ ]:
tf.download_data(
    domain="Global Q",
    dataset="q5_factors_monthly_2024",
    start_date=start_date,
    end_date=end_date,
)

date,risk_free,mkt_excess,me,ia,roe,eg
date,f64,f64,f64,f64,f64,f64
1967-01-01,0.003927,0.081852,0.068122,-0.029263,0.018813,-0.025511
1967-02-01,0.003743,0.007557,0.016235,-0.002915,0.035399,0.021792
1967-03-01,0.003693,0.040169,0.019836,-0.016772,0.018417,-0.011192
1967-04-01,0.003344,0.038786,-0.0067,-0.028972,0.010253,-0.016371
1967-05-01,0.003126,-0.042807,0.027457,0.021864,0.005901,0.001191
…,…,…,…,…,…,…
2024-08-01,0.004419,0.016518,-0.040817,0.004687,0.018369,0.008116
2024-09-01,0.004619,0.016806,-0.011967,-0.00001,0.007408,-0.03281
2024-10-01,0.003907,-0.009701,-0.011261,-0.011676,-0.002314,-0.008335


# Predictores Macroeconómicos: Welch y Goyal(2008)

In [ ]:
sheet_id = "1bM7vCWd3WOt95Sf9qjLPZjoiafgF_8EG"
sheet_name = "macro_predictors.xlsx"
macro_predictors_link = (
    f"https://docs.google.com/spreadsheets/d/{sheet_id}"
    f"/gviz/tq?tqx=out:csv&sheet={sheet_name}"
)

In [ ]:
macro_predictors = (
    pl.read_csv(requests.get(macro_predictors_link).content, infer_schema=False)
    .with_columns(
        pl.exclude("yyyymm").str.replace_all(",", "").cast(pl.Float64, strict=False)
    )
    .with_columns(index_div=pl.col("Index") + pl.col("D12"))
    .with_columns(
        logret=pl.col("index_div").log() - pl.col("index_div").shift(1).log()
    )
    .with_columns(
        date=pl.col("yyyymm").str.to_date("%Y%m"),
        rp_div=(pl.col("logret") - (pl.col("Rfree") + 1).log()).shift(-1),
        dp=pl.col("D12").log() - pl.col("Index").log(),
        dy=pl.col("D12").log() - pl.col("Index").shift(1).log(),
        ep=pl.col("E12").log() - pl.col("Index").log(),
        de=pl.col("D12").log() - pl.col("E12").log(),
        tms=pl.col("lty") - pl.col("tbl"),
        dfy=pl.col("BAA") - pl.col("AAA"),
    )
    .rename({"b/m": "bm"})
    .select(
        "date",
        "rp_div",
        "dp",
        "dy",
        "ep",
        "de",
        "svar",
        "bm",
        "ntis",
        "tbl",
        "lty",
        "ltr",
        "tms",
        "dfy",
        "infl",
    )
    .filter(
        (pl.col("date") >= pl.lit(start_date).str.to_date())
        & (pl.col("date") <= pl.lit(end_date).str.to_date())
    )
    .drop_nulls()
)
macro_predictors

date,rp_div,dp,dy,ep,de,svar,bm,ntis,tbl,lty,ltr,tms,dfy,infl
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1960-01-01,0.006569,-3.394191,-3.468337,-2.797533,-0.596658,0.000919,0.499502,0.022116,0.0435,0.0441,0.0112,0.0006,0.0073,-0.003401
1960-02-01,-0.016387,-3.383903,-3.374774,-2.806662,-0.577241,0.00115,0.493557,0.024037,0.0396,0.0429,0.0204,0.0033,0.0078,0.003413
1960-03-01,-0.018919,-3.350808,-3.364804,-2.792666,-0.558142,0.000969,0.549798,0.025593,0.0331,0.0411,0.0282,0.008,0.0076,0.0
1960-04-01,0.022958,-3.331425,-3.349108,-2.787838,-0.543587,0.000645,0.563404,0.025577,0.0323,0.0426,-0.017,0.0103,0.0075,0.003401
1960-05-01,0.016349,-3.356176,-3.329677,-2.827389,-0.528786,0.000424,0.541966,0.024414,0.0329,0.0417,0.0152,0.0088,0.0082,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2023-07-01,-0.022055,-4.198545,-4.167881,-3.226908,-0.971637,0.000537,0.205917,-0.016677,0.0525,0.039,-0.0035,-0.0135,0.0108,0.001908
2023-08-01,-0.053418,-4.17778,-4.195656,-3.203119,-0.974662,0.001337,0.210885,-0.016514,0.053,0.0417,-0.0052,-0.0113,0.0107,0.004367
2023-09-01,-0.026479,-4.124953,-4.1749,-3.147294,-0.97766,0.001023,0.218528,-0.017989,0.0532,0.0438,-0.0221,-0.0094,0.0103,0.002485


### Utilizando la libreria tidyfinance

In [ ]:
tf.download_data(
    domain="Goyal-Welch",
    dataset="monthly",
    start_date=start_date,
    end_date=end_date,
)

,date,rp_div,dp,dy,ep,de,svar,bm,ntis,tbl,lty,ltr,tms,dfy,infl
0,1960-01-01,0.006165,-3.394191,-3.468337,-2.797533,-0.596658,0.000919,0.499502,0.022116,0.0435,0.0441,0.0112,0.0006,0.0073,-0.003401
1,1960-02-01,-0.015793,-3.383903,-3.374774,-2.806662,-0.577241,0.001150,0.493557,0.024037,0.0396,0.0429,0.0204,0.0033,0.0078,0.003413
2,1960-03-01,-0.020521,-3.350808,-3.364804,-2.792666,-0.558142,0.000969,0.549798,0.025593,0.0331,0.0411,0.0282,0.0080,0.0076,0.000000
3,1960-04-01,0.023755,-3.331425,-3.349108,-2.787838,-0.543587,0.000645,0.563404,0.025577,0.0323,0.0426,-0.0170,0.0103,0.0075,0.003401
4,1960-05-01,0.016046,-3.356176,-3.329677,-2.827389,-0.528786,0.000424,0.541966,0.024414,0.0329,0.0417,0.0152,0.0088,0.0082,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
762,2023-07-01,-0.022065,-4.198545,-4.167881,-3.226908,-0.971637,0.000537,0.205917,-0.016677,0.0525,0.0390,-0.0035,-0.0135,0.0108,0.001908
763,2023-08-01,-0.053627,-4.177780,-4.195656,-3.203119,-0.974662,0.001337,0.210885,-0.016514,0.0530,0.0417,-0.0052,-0.0113,0.0107,0.004367
764,2023-09-01,-0.026090,-4.124953,-4.174900,-3.147294,-0.977660,0.001023,0.218528,-0.017989,0.0532,0.0438,-0.0221,-0.0094,0.0103,0.002485
765,2023-10-01,0.079457,-4.097976,-4.120201,-3.110379,-0.987598,0.001678,0.221534,-0.014711,0.0534,0.0480,-0.0121,-0.0054,0.0102,-0.000383


# Instituciones

## La Reserva Federal (FED)

In [ ]:
series = "CPIAUCNS" # Datos de inflación
url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"

In [ ]:
resp = requests.get(url, impersonate="chrome")

cpi_monthly = (
    pl.read_csv(resp.content)
    .with_columns(
        date=pl.col("observation_date").str.to_date(),
        value=pl.col(series).cast(pl.Float64, strict=False),
        series=pl.lit(series),
    )
    .select("date", "series", "value")
    .filter(
        (pl.col("date") >= pl.lit(start_date).str.to_date())
        & (pl.col("date") <= pl.lit(end_date).str.to_date())
    )
    .with_columns(cpi=pl.col("value") / pl.col("value").last())
)
cpi_monthly

date,series,value,cpi
date,str,f64,f64
1960-01-01,"""CPIAUCNS""",29.3,0.092838
1960-02-01,"""CPIAUCNS""",29.4,0.093154
1960-03-01,"""CPIAUCNS""",29.4,0.093154
1960-04-01,"""CPIAUCNS""",29.5,0.093471
1960-05-01,"""CPIAUCNS""",29.5,0.093471
…,…,…,…
2024-08-01,"""CPIAUCNS""",314.796,0.997437
2024-09-01,"""CPIAUCNS""",315.301,0.999037
2024-10-01,"""CPIAUCNS""",315.664,1.000187


### Utilizando la libreria tidyfinance

In [ ]:
tf.download_data(
    domain="FRED", series="CPIAUCNS", start_date=start_date, end_date=end_date
)

,date,series,value
0,1960-01-01,CPIAUCNS,29.300
1,1960-02-01,CPIAUCNS,29.400
2,1960-03-01,CPIAUCNS,29.400
3,1960-04-01,CPIAUCNS,29.500
4,1960-05-01,CPIAUCNS,29.500
...,...,...,...
775,2024-08-01,CPIAUCNS,314.796
776,2024-09-01,CPIAUCNS,315.301
777,2024-10-01,CPIAUCNS,315.664
778,2024-11-01,CPIAUCNS,315.493
